In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "workspace.default.pinterest_post_tags_silver"
GOLD_TABLE   = "workspace.default.pinterest_tag_counts_gold"

silver = spark.table(SILVER_TABLE)

silver_clean = (
    silver
    .where(F.col("post_year").isNotNull())
    .where(F.col("post_quarter").isNotNull())
    .where(F.col("tag").isNotNull())
    .withColumn("tag_norm", F.lower(F.trim(F.col("tag"))))
    .where(F.length("tag_norm") > 1)
)

gold = (
    silver_clean
    .groupBy("post_year", "post_quarter", F.col("tag_norm").alias("tag"))
    .agg(
        F.count("*").alias("tag_count"),
        F.countDistinct("user_id").alias("distinct_users")
    )
)

(
  gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)   # overwrites only touched partitions
)